# 07_TrainWide — Train 3 bản TCN với góc xoay rộng hơn (Task 6b)

Notebook này **không sửa 04_Trainv2** (Task 4 đã chốt). Nó lấy nguyên cách đọc dữ liệu, kiến trúc TCN, lật gương, early stopping, tối đa 60 epoch của 04, và chỉ đổi **khoảng góc xoay** (và mức nhiễu ở bản thứ 3). Chỉ train **TCN**.

| Bản | Xoay x / y / z | Nhiễu |
|---|---|---|
| `wide30` | ±30° / ±30° / ±30° | 0.01 |
| `wide45` | ±40° / ±45° / ±40° | 0.01 |
| `wide45_lownoise` | ±40° / ±45° / ±40° | 0.005 |

Checkpoint lưu ở `Data_321/checkpoints_v2/<tên bản>/TCN.pt`, **không đè** bản `all`. Train xong, cell 7 so cả 3 bản với `v2-TCN` hiện tại.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 7. Cell 4 là cell train (lâu nhất); nếu đã train rồi, chạy lại sẽ tự nạp checkpoint, không train lại.

In [1]:
#cell 1: đọc manifest và dựng tensor train/val/test giống 04_Trainv2 (bắt buộc CUDA)
import numpy as np, pandas as pd, torch, math, time
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
assert (BASE / "manifest.csv").exists(), f"Không thấy {BASE / 'manifest.csv'}"
m = pd.read_csv(BASE / "manifest.csv")

def feat(f):
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]                                           # lấy cổ tay làm gốc
    scale = np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    lm = lm / scale                                               # chuẩn hóa theo cỡ bàn tay
    pos = lm.reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])
    has = ~np.isnan(pos[:, 0])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

Tmax = int(m.khung.max())
def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32)
    M = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = feat(f)
        X[i, :len(x)] = x
        M[i, :len(x)] = h
    return (torch.tensor(X, device=dev),
            torch.tensor(df.nhan.values, device=dev),
            torch.tensor(M, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}
print("CUDA:", torch.cuda.get_device_name(0), "| Tmax:", Tmax)
for t, (X, y, M) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(M.sum().item()))

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (151, 355, 126) | video: 151 | khung có tay: 19108
val (31, 355, 126) | video: 31 | khung có tay: 3782
test (33, 355, 126) | video: 33 | khung có tay: 3930


In [2]:
#cell 2: kiến trúc TCN (giống hệt 02_Train / 04_Trainv2)
import torch.nn as nn, torch.nn.functional as F

class Block(nn.Module):
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c)
        s.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))

print("Đã định nghĩa TCN.")

Đã định nghĩa TCN.


In [3]:
#cell 3: cấu hình 3 bản + augmentation (xoay 3D, lật gương, nhiễu), đổi duy nhất khoảng góc và mức nhiễu so với 04
VARIANTS = {
    "wide30":          {"rot": (30, 30, 30), "noise": 0.01},    # độ, theo trục x / y / z
    "wide45":          {"rot": (40, 45, 40), "noise": 0.01},
    "wide45_lownoise": {"rot": (40, 45, 40), "noise": 0.005},
}

def augment_batch(x, hand_mask, rot_deg, noise_std):
    b, T = x.shape[0], x.shape[1]
    pos = x[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]

    # Mỗi clip một góc xoay ngẫu nhiên nhất quán theo thời gian (tọa độ đã lấy cổ tay làm gốc)
    rx, ry, rz = [math.radians(d) for d in rot_deg]
    ax = (torch.rand(b, 1, 1, device=x.device) * 2 - 1) * rx
    ay = (torch.rand(b, 1, 1, device=x.device) * 2 - 1) * ry
    az = (torch.rand(b, 1, 1, device=x.device) * 2 - 1) * rz
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)

    flip = torch.rand(b, 1, 1, device=x.device) < 0.5            # lật gương, giữ như 04
    pos[..., 0] = torch.where(flip, -pos[..., 0], pos[..., 0])

    pos = pos + torch.randn_like(pos) * noise_std * valid        # nhiễu tọa độ

    # Không làm thay đổi các khung mất tay/khung đệm; tính lại vận tốc sau augmentation.
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None, None]
    return torch.cat([pos.reshape(b, T, 63), vel.reshape(b, T, 63)], dim=-1)

for k, v in VARIANTS.items():
    print(f"{k:16s} | xoay x/y/z ±{v['rot']}° | nhiễu {v['noise']} | checkpoint: {BASE / 'checkpoints_v2' / k / 'TCN.pt'}")

wide30           | xoay x/y/z ±(30, 30, 30)° | nhiễu 0.01 | checkpoint: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\wide30\TCN.pt
wide45           | xoay x/y/z ±(40, 45, 40)° | nhiễu 0.01 | checkpoint: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\wide45\TCN.pt
wide45_lownoise  | xoay x/y/z ±(40, 45, 40)° | nhiễu 0.005 | checkpoint: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\wide45_lownoise\TCN.pt


In [4]:
#cell 4: train lần lượt 3 bản (CUDA, early stopping, mỗi bản một checkpoint riêng, nạp lại nếu đã có)
def loss_acc(net, X, y, M):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * M).sum() / M.sum().clamp(min=1), (correct & M).sum() / M.sum().clamp(min=1)

FORCE_RETRAIN = False   # True nếu muốn train lại tất cả 3 bản

def train_variant(name, cfg, epochs=60, bs=16, patience=10, force=False):
    torch.manual_seed(0)
    net = TCN().to(dev)
    path = BASE / "checkpoints_v2" / name / "TCN.pt"
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not force:
        ck = torch.load(path, map_location=dev, weights_only=False)
        if ck.get("cfg") == cfg:
            net.load_state_dict(ck["w"])
            print(f"{name}: nạp checkpoint có sẵn | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return net.eval()
        print(f"{name}: checkpoint cũ có cấu hình khác, train lại.")

    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, epochs + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); total = 0.0
        for i in range(0, len(Xt), bs):
            ids = perm[i:i + bs]
            xb, yb, mb = Xt[ids], yt[ids], Mt[ids]
            xb = augment_batch(xb, mb, cfg["rot"], cfg["noise"])
            loss, _ = loss_acc(net, xb, yb, mb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, *data["val"])
        row = {"ep": ep, "train_loss": total / len(Xt), "val_loss": vl.item(), "val_acc": va.item()}
        hist.append(row)
        if row["val_loss"] < best:
            best, wait = row["val_loss"], 0
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "hist": hist,
                        "model": "TCN", "variant": name, "cfg": cfg}, path)
        else:
            wait += 1
        if ep % 5 == 0 or wait >= patience or ep == epochs:
            print(f"{name} ep{ep:02d} | train {row['train_loss']:.4f} | val {row['val_loss']:.4f} | val_acc {row['val_acc']:.3f}")
        if wait >= patience:
            print(f"Dừng sớm: val loss không giảm sau {patience} epoch."); break
    ck = torch.load(path, map_location=dev, weights_only=False)
    net.load_state_dict(ck["w"])
    print(f"{name}: tốt nhất epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f} | {time.time() - t0:.0f}s")
    return net.eval()

wide_nets = {}
for name, cfg in VARIANTS.items():
    wide_nets[name] = train_variant(name, cfg, force=FORCE_RETRAIN)
    print("-" * 70)
print("Hoàn tất train 3 bản.")

wide30 ep05 | train 0.6744 | val 0.4802 | val_acc 0.911
wide30 ep10 | train 0.4071 | val 0.3232 | val_acc 0.896
wide30 ep15 | train 0.3246 | val 0.3014 | val_acc 0.901
wide30 ep20 | train 0.2967 | val 0.3571 | val_acc 0.847
wide30 ep25 | train 0.3981 | val 0.3009 | val_acc 0.893
wide30 ep30 | train 0.2634 | val 0.2772 | val_acc 0.865
wide30 ep35 | train 0.2678 | val 0.2891 | val_acc 0.882
wide30 ep40 | train 0.2911 | val 0.2907 | val_acc 0.902
wide30 ep45 | train 0.2273 | val 0.2515 | val_acc 0.898
wide30 ep50 | train 0.2043 | val 0.2278 | val_acc 0.906
wide30 ep52 | train 0.2133 | val 0.3094 | val_acc 0.883
Dừng sớm: val loss không giảm sau 10 epoch.
wide30: tốt nhất epoch 42 | val_loss 0.1951 | 4s
----------------------------------------------------------------------
wide45 ep05 | train 0.8085 | val 0.6160 | val_acc 0.815
wide45 ep10 | train 0.4387 | val 0.3936 | val_acc 0.866
wide45 ep15 | train 0.3414 | val 0.2972 | val_acc 0.868
wide45 ep20 | train 0.3154 | val 0.3436 | val_acc 0.

In [5]:
#cell 5: nạp v2-TCN hiện tại (chế độ "all") để so sánh cùng 3 bản wide
p_all = BASE / "checkpoints_v2" / "all" / "TCN.pt"
assert p_all.exists(), f"Không thấy {p_all}: cần checkpoint v2-TCN từ 04_Trainv2 (AUGMENT_MODE='all')."
net_all = TCN().to(dev)
net_all.load_state_dict(torch.load(p_all, map_location=dev, weights_only=False)["w"])

nets = {"v2-TCN": net_all.eval(), **{f"{k}": v for k, v in wide_nets.items()}}
print("Sẽ so sánh:", list(nets))

Sẽ so sánh: ['v2-TCN', 'wide30', 'wide45', 'wide45_lownoise']


In [6]:
#cell 6: hàm xoay cố định/ngẫu nhiên khi đánh giá + chỉ số (giống 06_AngleCheck)
Xte, yte, Mte = data["test"]
THR = 0.7

def rotate_clip(x, hand_mask, ax_deg, ay_deg, az_deg):
    b, T = x.shape[0], x.shape[1]
    pos = x[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    ax, ay, az = [torch.as_tensor(a, dtype=torch.float32, device=x.device) * math.pi / 180
                  for a in (ax_deg, ay_deg, az_deg)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None, None]
    return torch.cat([pos.reshape(b, T, 63), vel.reshape(b, T, 63)], dim=-1)

def evaluate(net, Xr):
    with torch.no_grad():
        conf, pred = net(Xr).softmax(-1).max(-1)
    ok = pred == yte[:, None]
    sel = (conf >= THR) & Mte
    n_all, n_sel = Mte.sum().item(), sel.sum().item()
    return {"acc_all": (ok & Mte).sum().item() / n_all * 100,
            "acc_conf": (ok & sel).sum().item() / n_sel * 100 if n_sel else float("nan"),
            "coverage": n_sel / n_all * 100}

err = (rotate_clip(Xte, Mte, 0, 0, 0) - Xte).abs().max().item()
print(f"Sai lệch tối đa khi xoay 0°: {err:.2e}")
assert err < 1e-4, "Xoay 0° không khớp dữ liệu gốc, kiểm tra lại hàm rotate_clip."

Sai lệch tối đa khi xoay 0°: 0.00e+00


In [7]:
#cell 7: so sánh v2-TCN với 3 bản wide — xoay cả 3 trục (±mag, mỗi video một góc ngẫu nhiên, lặp REPEAT lần), cùng dữ liệu xoay cho mọi model
MAGS = [0, 15, 30, 45, 60]
REPEAT = 5
g_rand = torch.Generator(device=dev).manual_seed(0)

rows = []
for mag in MAGS:
    acc = {name: [] for name in nets}
    for _ in range(1 if mag == 0 else REPEAT):
        if mag == 0:
            Xr = Xte
        else:
            rnd = lambda: (torch.rand(len(Xte), 1, 1, device=dev, generator=g_rand) * 2 - 1) * mag
            Xr = rotate_clip(Xte, Mte, rnd(), rnd(), rnd())
        for name, net in nets.items():
            acc[name].append(evaluate(net, Xr))
    for name, lst in acc.items():
        rows.append({"model": name, "mag": mag,
                     "acc_all": np.nanmean([r["acc_all"] for r in lst]),
                     "acc_conf": np.nanmean([r["acc_conf"] for r in lst]),
                     "coverage": np.nanmean([r["coverage"] for r in lst])})
res = pd.DataFrame(rows)

for metric, title in [("acc_all", "Độ chính xác mọi khung có tay (%)"),
                      ("acc_conf", f"Độ chính xác khi tự tin ≥ {THR} (%)"),
                      ("coverage", f"Độ phủ: % khung đạt ngưỡng {THR}")]:
    print(title, "— cột là ±mag (độ), 0 = không xoay")
    display(res.pivot_table(index="model", columns="mag", values=metric).reindex(list(nets)).round(1))

# Tiêu chí chọn bản: 0° không tụt quá 1 điểm so với v2-TCN, và tốt hơn v2-TCN ở ±45°
p = res.pivot_table(index="model", columns="mag", values="acc_all").reindex(list(nets))
verdict = pd.DataFrame({"acc 0°": p[0], "Δ0° vs v2": p[0] - p.loc["v2-TCN", 0],
                        "acc ±45°": p[45], "Δ±45° vs v2": p[45] - p.loc["v2-TCN", 45]})
verdict["đạt 0° (không tụt >1)"] = verdict["Δ0° vs v2"] >= -1
verdict["tốt hơn ở ±45°"] = verdict["Δ±45° vs v2"] > 0
print("Tóm tắt theo tiêu chí chọn bản:")
display(verdict.round(1))

Độ chính xác mọi khung có tay (%) — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
v2-TCN,91.9,91.9,90.7,84.4,86.0
wide30,92.0,91.9,91.8,90.0,88.6
wide45,91.9,92.2,91.4,90.4,88.5
wide45_lownoise,92.5,92.3,91.3,91.2,90.4


Độ chính xác khi tự tin ≥ 0.7 (%) — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
v2-TCN,98.9,98.8,98.1,91.1,91.5
wide30,97.8,97.7,97.4,95.8,95.1
wide45,97.7,97.6,97.1,96.4,95.6
wide45_lownoise,96.7,96.7,97.2,97.7,96.4


Độ phủ: % khung đạt ngưỡng 0.7 — cột là ±mag (độ), 0 = không xoay


mag,0,15,30,45,60
model,,,,,
v2-TCN,81.8,81.5,80.1,76.9,79.2
wide30,87.6,87.6,86.6,85.3,82.2
wide45,88.0,88.2,85.7,84.3,81.8
wide45_lownoise,89.4,89.2,87.2,85.9,84.6


Tóm tắt theo tiêu chí chọn bản:


,acc 0°,Δ0° vs v2,acc ±45°,Δ±45° vs v2,đạt 0° (không tụt >1),tốt hơn ở ±45°
model,,,,,,
v2-TCN,91.9,0.0,84.4,0.0,True,False
wide30,92.0,0.2,90.0,5.7,True,True
wide45,91.9,0.0,90.4,6.0,True,True
wide45_lownoise,92.5,0.6,91.2,6.9,True,True


## Cách đọc kết quả
- Bản nào **đạt cả 2 tiêu chí** (cột `đạt 0°` và `tốt hơn ở ±45°` cùng True) là ứng viên để test webcam bằng notebook 05.
- Tập test chỉ có 33 video: chênh lệch dưới khoảng 2 điểm giữa các bản **không đáng tin**. Bản thắng phải được xác nhận lại trên webcam trước khi chốt.
- Nếu không bản nào đạt, hoặc bản wide làm 0° tụt rõ, nghĩa là mở rộng góc xoay không giúp và ta đổi hướng (Task 6 sẽ ghi lại).
- Chép bảng tóm tắt ở cell 7 vào phần Ghi chép tiến trình (chỉ cần một hai dòng kết quả).